# Selfie2Anime - CycleGAN (PyTorch)
Convert your selfie photos into anime-style images using a pre-trained CycleGAN model.

**Instructions:**
1. Run all cells in order
2. Upload your selfie when prompted
3. See the anime-style output!

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/Parth-Vasave/selfie2animePyTorch.git
%cd selfie2animePyTorch/cycle_gan
!pip install -q torch torchvision opencv-python matplotlib Pillow numpy

## 2. Load the model

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pix2pix_gan import ResnetGenerator
from dataset import reconstruct_color

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = ResnetGenerator().to(device)
model.load_state_dict(torch.load('model/selfie2anime.gen_ab.pth', map_location=device, weights_only=True))
model.eval()
print('Model loaded successfully!')

## 3. Upload your selfie

In [ ]:
from google.colab import files

uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
print(f'Uploaded: {input_filename}')

## 4. Convert selfie to anime

In [ ]:
def preprocess(img_np, size=256):
    h, w = img_np.shape[:2]
    if h < w:
        new_h, new_w = size, int(w * size / h)
    else:
        new_h, new_w = int(h * size / w), size
    img_np = cv2.resize(img_np, (new_w, new_h))
    tensor = torch.from_numpy(img_np).permute(2, 0, 1).float() / 255.0
    tensor = (tensor - 0.5) / 0.5
    return tensor.unsqueeze(0)

# Read and convert
raw = cv2.imread(input_filename)
raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)

input_tensor = preprocess(raw).to(device)

with torch.no_grad():
    fake, _ = model(input_tensor)

anime_img = reconstruct_color(fake[0].permute(1, 2, 0).cpu()).numpy()

# Display results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
ax1.imshow(raw)
ax1.set_title('Original Selfie')
ax1.axis('off')
ax2.imshow(anime_img)
ax2.set_title('Anime Style')
ax2.axis('off')
plt.tight_layout()
plt.show()

## 5. Download the result

In [ ]:
from PIL import Image

output_filename = 'anime_' + input_filename
Image.fromarray(anime_img).save(output_filename)
files.download(output_filename)
print(f'Saved and downloading: {output_filename}')